Evaluator Optimizer

In the evaluator-optimizer workflow, one LLM call generates a response while another provides evaluation and feedback in a loop.

When to use this workflow:
This workflow is particularly effective when we have clear evaluation criteria, and when iterative refinement provides measurable value. The two signs of good fit are:

LLM responses can be demonstrably improved when a human articulates their feedback.
The LLM can provide such feedback.

This is analogous to the iterative writing process a human writer might go through when producing a polished document.

Fir generator ko hi bol do

"Generate and verify"

Kar sakte hain.

Example

Generate a blog.

Then verify it.

Then improve it.

Ye bhi kaam karta hai.

Lekin problem hai.

Self-review bias

LLM apni hi reasoning ko repeat karta hai.

Suppose pehli baar usne galat fact likha.

Second step me bhi wahi reasoning use karega.

To kabhi-kabhi same mistake ko correct nahi karta.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_groq import ChatGroq

# os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="qwen-2.5-32b")
result=llm.invoke("Write a short story about a robot learning to love.")

In [ ]:
from typing import Annotated, List
import operator
from typing_extensions import Literal
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage, SystemMessage
from typing_extensions import TypedDict

# Graph state
class State(TypedDict):
    joke: str
    topic: str
    feedback: str
    funny_or_not: str

In [ ]:
# Schema for structured output to use in evaluation
class Feedback(BaseModel):
    grade: Literal["funny", "not funny"] = Field(
        description="Decide if the joke is funny or not.",
    )

    feedback: str = Field(
        description="If the joke is not funny, provide feedback on how to improve it.",
    )

evaluator=llm.with_structured_output(Feedback)

In [ ]:
# Nodes
def llm_call_generator(state: State):
    """LLM generates a joke"""

    if state.get("feedback"):
        msg = llm.invoke(
            f"Write a joke about {state['topic']} but take into account the feedback: {state['feedback']}"
        )
    else:
        msg = llm.invoke(
            f"Write a joke about {state['topic']}"
        )

    return {
        "joke": msg.content
    }

def llm_call_evaluator(state: State):
    """LLM evaluates the joke"""

    grade = evaluator.invoke(
        f"Grade the joke {state['joke']}"
    )

    return {
        "funny_or_not": grade.grade,
        "feedback": grade.feedback
    }

# Conditional edge function to route back to joke generator or end
# based upon feedback from the evaluator

def route_joke(state: State):
    """Route back to joke generator or end based upon feedback from the evaluator"""

    if state["funny_or_not"] == "funny":
        return "Accepted"

    elif state["funny_or_not"] == "not funny":
        return "Rejected + Feedback"

In [ ]:
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display

In [ ]:
# Build workflow
optimizer_builder = StateGraph(State)

# Add the nodes
optimizer_builder.add_node("llm_call_generator", llm_call_generator)
optimizer_builder.add_node("llm_call_evaluator", llm_call_evaluator)

# Add edges to connect nodes
optimizer_builder.add_edge(START, "llm_call_generator")
optimizer_builder.add_edge("llm_call_generator", "llm_call_evaluator")

optimizer_builder.add_conditional_edges(
    "llm_call_evaluator",
    route_joke,
    {
        # Name returned by route_joke : Name of next node to visit
        "Accepted": END,
        "Rejected + Feedback": "llm_call_generator",
    },
)

optimizer_workflow=optimizer_builder.compile()